Rule-based Segmented Forecasting Model

Notebook này tạo dự báo 56 ngày cho từng SKU bằng mô hình rule-based có phân nhóm. Mô hình ưu tiên chống over-forecast cho SKU bán thưa, đồng thời vẫn giữ tín hiệu tuần và nhóm SKU có lợi nhuận cao.

## 1. Cấu hình

Thiết lập seed, đường dẫn dữ liệu, horizon dự báo và số fold validation.

In [ ]:
import os
import gc
import json
import random
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR_CANDIDATES = [Path("dataset"), Path("."), Path(".."), Path("/mnt/data")]
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE_NAME = "train_cleaned.csv"
SAMPLE_FILE_NAME = "sample_submission.csv"

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
VALID_LOG_PATH = OUTPUT_DIR / "validation_log.csv"
BEST_CONFIG_PATH = OUTPUT_DIR / "best_config.json"

FORECAST_HORIZON = 56
PUBLIC_HORIZON = 28

VALID_HORIZON = 56
VALID_FOLD_OFFSETS = [56, 84, 112]

# True: grid gọn để chạy nhanh. False: grid rộng hơn, chạy lâu hơn.
FAST_SEARCH = True

print("SEED:", SEED)
print("SUBMISSION_PATH:", SUBMISSION_PATH)
print("VALID_LOG_PATH:", VALID_LOG_PATH)
print("BEST_CONFIG_PATH:", BEST_CONFIG_PATH)


## 2. Tìm file đầu vào

Notebook cần `train_cleaned.csv` và `sample_submission.csv`.

In [ ]:
def find_file(file_name: str, required: bool = True):
    for folder in DATA_DIR_CANDIDATES:
        path = folder / file_name
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(
            f"Không tìm thấy {file_name}. Hãy đặt file trong dataset/ hoặc cùng folder notebook."
        )
    return None

TRAIN_PATH = find_file(TRAIN_FILE_NAME)
SAMPLE_PATH = find_file(SAMPLE_FILE_NAME)

print("TRAIN_PATH:", TRAIN_PATH)
print("SAMPLE_PATH:", SAMPLE_PATH)


## 3. Load dữ liệu

Đọc dữ liệu clean, chuẩn hóa kiểu dữ liệu và lấy danh sách SKU từ sample submission để không thiếu SKU cần dự báo.

In [ ]:
usecols = ["Date", "ItemCode", "Quantity", "SalesAmount", "Cost_Amount"]
train = pd.read_csv(TRAIN_PATH, usecols=usecols)
sample_sub = pd.read_csv(SAMPLE_PATH)

train["Date"] = pd.to_datetime(train["Date"])
train["Quantity"] = pd.to_numeric(train["Quantity"], errors="coerce").fillna(0).astype("float32")
train["SalesAmount"] = pd.to_numeric(train["SalesAmount"], errors="coerce").fillna(0).astype("float32")
train["Cost_Amount"] = pd.to_numeric(train["Cost_Amount"], errors="coerce").fillna(0).astype("float32")
train["Profit"] = (train["SalesAmount"] - train["Cost_Amount"]).astype("float32")

all_skus = (
    sample_sub["id"]
    .astype(str)
    .str.replace("_validation", "", regex=False)
    .str.replace("_evaluation", "", regex=False)
    .unique()
)
all_skus = np.array(sorted(all_skus))
forecast_cols = [f"F{i}" for i in range(1, PUBLIC_HORIZON + 1)]

print("Train shape:", train.shape)
print("Sample shape:", sample_sub.shape)
print("SKU in sample:", len(all_skus))
print("SKU in train:", train["ItemCode"].nunique())
print("Date range:", train["Date"].min(), "→", train["Date"].max())
display(train.head())
display(sample_sub.head())

## 4. Tạo daily matrix

Gom transaction theo `Date × ItemCode`, sau đó tạo matrix `SKU × Date` cho Quantity và Profit. Quantity âm sau khi gom ngày được clip về 0.

In [ ]:
daily = (
    train.groupby(["Date", "ItemCode"], as_index=False)
    .agg(Quantity=("Quantity", "sum"), Profit=("Profit", "sum"))
)

daily["Quantity_Raw"] = daily["Quantity"].astype("float32")
daily["Quantity"] = daily["Quantity"].clip(lower=0).astype("float32")
daily["Profit"] = daily["Profit"].astype("float32")

all_dates = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")
N_ITEMS = len(all_skus)
N_DATES = len(all_dates)

qty_df = (
    daily.pivot(index="Date", columns="ItemCode", values="Quantity")
    .reindex(index=all_dates, columns=all_skus, fill_value=0)
    .fillna(0)
    .astype("float32")
)
profit_df = (
    daily.pivot(index="Date", columns="ItemCode", values="Profit")
    .reindex(index=all_dates, columns=all_skus, fill_value=0)
    .fillna(0)
    .astype("float32")
)

QTY = qty_df.to_numpy(dtype=np.float32).T
PROFIT = profit_df.to_numpy(dtype=np.float32).T

neg_daily_rows = int((daily["Quantity_Raw"] < 0).sum())
positive_ratio = float((QTY > 0).sum() / QTY.size)

print("N_ITEMS:", N_ITEMS)
print("N_DATES:", N_DATES)
print("QTY shape:", QTY.shape)
print("PROFIT shape:", PROFIT.shape)
print("Negative daily rows before clip:", neg_daily_rows)
print("Positive demand ratio:", positive_ratio)

del qty_df, profit_df, daily
gc.collect()

## 5. Precompute lịch sử

Tạo cumulative arrays để tính rolling mean, số ngày bán và profit-to-date nhanh, không dùng dữ liệu tương lai.

In [ ]:
QTY_CSUM = np.concatenate(
    [np.zeros((N_ITEMS, 1), dtype=np.float32), np.cumsum(QTY, axis=1, dtype=np.float32)],
    axis=1,
)

SALES_DAY_CSUM = np.concatenate(
    [np.zeros((N_ITEMS, 1), dtype=np.float32), np.cumsum((QTY > 0).astype(np.float32), axis=1, dtype=np.float32)],
    axis=1,
)

PROFIT_CSUM = np.concatenate(
    [np.zeros((N_ITEMS, 1), dtype=np.float32), np.cumsum(PROFIT, axis=1, dtype=np.float32)],
    axis=1,
)

LAST_POS_IDX = np.full((N_ITEMS, N_DATES), -1, dtype=np.int16)
last = np.full(N_ITEMS, -1, dtype=np.int16)
for d in range(N_DATES):
    sold_today = QTY[:, d] > 0
    last[sold_today] = d
    LAST_POS_IDX[:, d] = last

print("Precompute done.")

## 6. Hàm tiện ích và WRMSSE

Các hàm bên dưới phục vụ tính rolling feature, weekday average, profit weight và WRMSSE local validation.

In [ ]:
def rolling_sum_at(csum: np.ndarray, origin_idx: int, window: int) -> np.ndarray:
    end = origin_idx + 1
    start = max(0, end - window)
    return csum[:, end] - csum[:, start]


def rolling_mean_at(csum: np.ndarray, origin_idx: int, window: int) -> np.ndarray:
    actual_window = min(window, origin_idx + 1)
    return rolling_sum_at(csum, origin_idx, window) / max(actual_window, 1)


def weekday_avg_for_target(origin_idx: int, horizon: int, n_weeks: int = 8) -> np.ndarray:
    target_idx = origin_idx + horizon
    candidate_indices = []
    k = 1
    while len(candidate_indices) < n_weeks:
        idx = target_idx - 7 * k
        if 0 <= idx <= origin_idx:
            candidate_indices.append(idx)
        if idx < 0:
            break
        k += 1
    if len(candidate_indices) == 0:
        return np.zeros(N_ITEMS, dtype=np.float32)
    return QTY[:, candidate_indices].mean(axis=1).astype(np.float32)


def get_top_profit_mask(origin_idx: int, top_profit_quantile: float):
    profit_to_date = PROFIT_CSUM[:, origin_idx + 1]
    profit_for_rank = np.maximum(profit_to_date, 0)
    positive_profit = profit_for_rank[profit_for_rank > 0]
    if len(positive_profit) == 0:
        return np.zeros(N_ITEMS, dtype=bool), profit_to_date
    threshold = np.quantile(positive_profit, top_profit_quantile)
    return profit_for_rank >= threshold, profit_to_date


def compute_rmsse_denominator(cutoff_idx: int) -> np.ndarray:
    hist = QTY[:, :cutoff_idx + 1]
    diffs = np.diff(hist, axis=1)
    denom = np.mean(diffs ** 2, axis=1)
    return np.where(denom > 1e-9, denom, 1.0).astype(np.float32)


def compute_profit_weights(cutoff_idx: int) -> np.ndarray:
    profit = PROFIT_CSUM[:, cutoff_idx + 1]
    profit = np.maximum(profit, 0)
    total = profit.sum()
    if total <= 0:
        return np.ones(N_ITEMS, dtype=np.float32) / N_ITEMS
    return (profit / total).astype(np.float32)


def wrmsse_score(y_true_matrix: np.ndarray, y_pred_matrix: np.ndarray, cutoff_idx: int) -> float:
    denom = compute_rmsse_denominator(cutoff_idx)
    weights = compute_profit_weights(cutoff_idx)
    return wrmsse_score_cached(y_true_matrix, y_pred_matrix, denom, weights)


def wrmsse_score_cached(y_true_matrix: np.ndarray, y_pred_matrix: np.ndarray, denom: np.ndarray, weights: np.ndarray) -> float:
    rmse = np.sqrt(np.mean((y_true_matrix - y_pred_matrix) ** 2, axis=1))
    rmsse = rmse / np.sqrt(denom)
    rmsse = np.nan_to_num(rmsse, nan=0.0, posinf=0.0, neginf=0.0)
    return float(np.sum(weights * rmsse))

print("Utility functions ready.")

## 7. Hàm dự báo

Mỗi SKU được chia nhóm theo mức độ hoạt động gần đây. Forecast kết hợp moving average, weekday pattern, sparse cap, active cap, stale rule và hiệu ứng Chủ nhật.

In [ ]:
def get_calendar_multiplier(config: dict, target_date: pd.Timestamp, horizon: int) -> float:
    """Calendar adjustment nhẹ, tất cả hệ số mặc định = 1.0 để fallback tương thích các bản cũ."""
    mult = 1.0

    weekend_multiplier = float(config.get("weekend_multiplier", 1.0))
    sunday_multiplier = float(config.get("sunday_multiplier", 1.0))
    month_start_end_multiplier = float(config.get("month_start_end_multiplier", 1.0))
    high_season_multiplier = float(config.get("high_season_multiplier", 1.0))
    eval_window_multiplier = float(config.get("eval_window_multiplier", 1.0))

    if target_date.weekday() >= 5:
        mult *= weekend_multiplier
    if target_date.weekday() == 6:
        mult *= sunday_multiplier
    if target_date.day in [1, 2, 3, 28, 29, 30, 31]:
        mult *= month_start_end_multiplier
    if target_date.month in [1, 10, 11, 12]:
        mult *= high_season_multiplier
    if horizon > 28:
        mult *= eval_window_multiplier

    return float(mult)


def forecast_for_origin(config: dict, origin_idx: int, horizon: int) -> np.ndarray:
    recent_window = int(config["recent_window"])
    sparse_threshold = int(config["sparse_threshold"])
    sparse_multiplier = float(config["sparse_multiplier"])
    top_profit_quantile = float(config["top_profit_quantile"])
    top_ma_window = int(config["top_ma_window"])
    top_ma_weight = float(config["top_ma_weight"])
    top_weekday_weight = 1.0 - top_ma_weight
    top_weekday_weeks = int(config.get("top_weekday_weeks", 8))
    other_mid_window = int(config["other_mid_window"])
    other_long_window = int(config["other_long_window"])
    other_mid_weight = float(config["other_mid_weight"])
    other_long_weight = 1.0 - other_mid_weight
    sparse_cap_window = int(config["sparse_cap_window"])
    sparse_cap_multiplier = float(config.get("sparse_cap_multiplier", 1.0))
    active_cap_window = int(config.get("active_cap_window", 56))
    active_cap_multiplier = float(config.get("active_cap_multiplier", 999.0))

    stale_days_threshold = int(config.get("stale_days_threshold", 9999))
    stale_multiplier = float(config.get("stale_multiplier", 1.0))
    very_stale_days_threshold = int(config.get("very_stale_days_threshold", 9999))
    very_stale_multiplier = float(config.get("very_stale_multiplier", 1.0))
    stale_apply_to_top_profit = int(config.get("stale_apply_to_top_profit", 1))
    low_recent_window = int(config.get("low_recent_window", 14))
    low_recent_threshold = int(config.get("low_recent_threshold", -1))
    low_recent_multiplier = float(config.get("low_recent_multiplier", 1.0))
    low_recent_apply_to_top_profit = int(config.get("low_recent_apply_to_top_profit", 1))

    total_recent = rolling_sum_at(QTY_CSUM, origin_idx, recent_window)
    sales_days_recent = rolling_sum_at(SALES_DAY_CSUM, origin_idx, recent_window)

    ma_top = rolling_mean_at(QTY_CSUM, origin_idx, top_ma_window)
    ma_mid = rolling_mean_at(QTY_CSUM, origin_idx, other_mid_window)
    ma_long = rolling_mean_at(QTY_CSUM, origin_idx, other_long_window)
    weekday_avg = weekday_avg_for_target(origin_idx, horizon, n_weeks=top_weekday_weeks)

    is_top_profit, _ = get_top_profit_mask(origin_idx, top_profit_quantile)

    pred = np.zeros(N_ITEMS, dtype=np.float32)

    mask_zero = sales_days_recent == 0
    mask_sparse = (sales_days_recent > 0) & (sales_days_recent <= sparse_threshold)
    mask_active_top = (sales_days_recent > sparse_threshold) & is_top_profit
    mask_active_other = (sales_days_recent > sparse_threshold) & (~is_top_profit)

    pred[mask_zero] = 0.0

    # Sparse: tiếp tục kiểm soát thấp quanh vùng V9 thắng.
    sparse_base = total_recent / max(recent_window, 1)
    sparse_cap = sparse_cap_multiplier * rolling_mean_at(QTY_CSUM, origin_idx, sparse_cap_window)
    sparse_pred = sparse_multiplier * sparse_base
    sparse_pred = np.minimum(sparse_pred, sparse_cap)
    pred[mask_sparse] = sparse_pred[mask_sparse]

    # Top-profit active: V8 cho thấy nên cân bằng hơn giữa MA 28 và weekday pattern.
    pred[mask_active_top] = (
        top_ma_weight * ma_top[mask_active_top]
        + top_weekday_weight * weekday_avg[mask_active_top]
    )

    # Active non-top: giữ neo ổn định 56/90.
    pred[mask_active_other] = (
        other_mid_weight * ma_mid[mask_active_other]
        + other_long_weight * ma_long[mask_active_other]
    )

    # Calendar multiplier áp dụng cho nhóm non-zero rule.
    target_date = pd.Timestamp(all_dates[origin_idx]) + pd.Timedelta(days=int(horizon))
    calendar_multiplier = get_calendar_multiplier(config, target_date, horizon)
    active_or_sparse_mask = mask_sparse | mask_active_top | mask_active_other
    pred[active_or_sparse_mask] *= calendar_multiplier

    # Stale rule: giảm forecast khi SKU không bán gần đây.
    if (stale_days_threshold < 9999 and stale_multiplier < 1.0) or (very_stale_days_threshold < 9999 and very_stale_multiplier < 1.0):
        last_pos = LAST_POS_IDX[:, origin_idx]
        days_since = np.where(last_pos >= 0, origin_idx - last_pos, 9999)

        stale_base_mask = active_or_sparse_mask
        if stale_apply_to_top_profit == 0:
            stale_base_mask = stale_base_mask & (~mask_active_top)

        if stale_days_threshold < 9999 and stale_multiplier < 1.0:
            stale_mask = stale_base_mask & (days_since >= stale_days_threshold)
            pred[stale_mask] *= stale_multiplier

        if very_stale_days_threshold < 9999 and very_stale_multiplier < 1.0:
            very_stale_mask = stale_base_mask & (days_since >= very_stale_days_threshold)
            pred[very_stale_mask] *= very_stale_multiplier

    # Low recent activity: nếu 14 ngày gần nhất có quá ít ngày bán, giảm thêm forecast.
    # Đây là tầng kiểm soát mới của V9, mặc định không kích hoạt nếu multiplier = 1.0.
    if low_recent_threshold >= 0 and low_recent_multiplier < 1.0:
        sales_days_low_recent = rolling_sum_at(SALES_DAY_CSUM, origin_idx, low_recent_window)
        low_recent_mask = active_or_sparse_mask & (sales_days_low_recent <= low_recent_threshold)
        if low_recent_apply_to_top_profit == 0:
            low_recent_mask = low_recent_mask & (~mask_active_top)
        pred[low_recent_mask] *= low_recent_multiplier

    # Active cap: chống forecast vọt, áp dụng sau stale/low_recent để vẫn giữ cap an toàn.
    if active_cap_multiplier < 100:
        active_mask = mask_active_top | mask_active_other
        active_cap = active_cap_multiplier * rolling_mean_at(QTY_CSUM, origin_idx, active_cap_window)
        pred[active_mask] = np.minimum(pred[active_mask], active_cap[active_mask])

    pred = np.clip(pred, 0, None).astype(np.float32)
    return pred


def forecast_matrix_for_origin(config: dict, origin_idx: int, horizon: int = FORECAST_HORIZON) -> np.ndarray:
    pred = np.zeros((N_ITEMS, horizon), dtype=np.float32)
    for h in range(1, horizon + 1):
        pred[:, h - 1] = forecast_for_origin(config, origin_idx, h)
    return pred

print("Forecast functions ready.")


## 8. Validation folds

Dùng các mốc validation lùi từ cuối train để mô phỏng forecast 56 ngày mà không leakage.

In [ ]:
train_end_idx = N_DATES - 1
valid_origins = []

for offset in VALID_FOLD_OFFSETS:
    origin_idx = train_end_idx - offset
    if origin_idx > 120 and origin_idx + VALID_HORIZON <= train_end_idx:
        valid_origins.append(origin_idx)

if len(valid_origins) == 0:
    raise ValueError("Không tạo được validation fold. Cần kiểm tra N_DATES hoặc VALID_FOLD_OFFSETS.")

print("Validation folds:")
for i, origin_idx in enumerate(valid_origins, start=1):
    print(
        f"Fold {i}: origin={all_dates[origin_idx].date()}, "
        f"target={all_dates[origin_idx+1].date()} → {all_dates[origin_idx+VALID_HORIZON].date()}"
    )

valid_cache = {}
for origin_idx in valid_origins:
    valid_cache[origin_idx] = {
        "y_true": QTY[:, origin_idx + 1: origin_idx + VALID_HORIZON + 1].astype(np.float32),
        "denom": compute_rmsse_denominator(origin_idx),
        "weights": compute_profit_weights(origin_idx),
    }

print("Validation cache ready.")

## 9. Grid search tham số

Grid tập trung vào các tham số quan trọng: sparse rule, active cap, stale rule, Sunday multiplier và low recent activity.

In [ ]:
PARAM_GRID_RAW = {
    "recent_window": [90],
    "sparse_threshold": [9, 10, 11],
    "sparse_multiplier": [0.18, 0.20, 0.22, 0.25],
    "top_profit_quantile": [0.80],
    "top_ma_window": [28],
    "top_ma_weight": [0.45, 0.50, 0.55],
    "top_weekday_weeks": [8],
    "other_mid_window": [56],
    "other_long_window": [90],
    "other_mid_weight": [0.70],
    "sparse_cap_window": [7],
    "sparse_cap_multiplier": [0.60, 0.80],
    "active_cap_window": [21],
    "active_cap_multiplier": [1.08, 1.10, 1.12],
    "weekend_multiplier": [1.00],
    "sunday_multiplier": [0.65, 0.70, 0.75],
    "month_start_end_multiplier": [1.00],
    "high_season_multiplier": [1.00],
    "eval_window_multiplier": [1.00],
    "stale_days_threshold": [2, 3, 5],
    "stale_multiplier": [0.05, 0.08, 0.10],
    "very_stale_days_threshold": [9999],
    "very_stale_multiplier": [1.00],
    "stale_apply_to_top_profit": [1],
    "low_recent_window": [14],
    "low_recent_threshold": [0, 1],
    "low_recent_multiplier": [0.85, 0.90, 1.00],
    "low_recent_apply_to_top_profit": [1],
}

if FAST_SEARCH:
    # Grid cân bằng khoảng 384 configs.
    # Giữ các trục quan trọng nhất nhưng loại bỏ các tổ hợp quá cực đoan.
    PARAM_GRID_RAW = {
        "recent_window": [90],
        "sparse_threshold": [9, 10],
        "sparse_multiplier": [0.20, 0.22],
        "top_profit_quantile": [0.80],
        "top_ma_window": [28],
        "top_ma_weight": [0.45, 0.50],
        "top_weekday_weeks": [8],
        "other_mid_window": [56],
        "other_long_window": [90],
        "other_mid_weight": [0.70],
        "sparse_cap_window": [7],
        "sparse_cap_multiplier": [0.60],
        "active_cap_window": [21],
        "active_cap_multiplier": [1.08, 1.10],
        "weekend_multiplier": [1.00],
        "sunday_multiplier": [0.65, 0.70, 0.75],
        "month_start_end_multiplier": [1.00],
        "high_season_multiplier": [1.00],
        "eval_window_multiplier": [1.00],
        "stale_days_threshold": [2, 3],
        "stale_multiplier": [0.05, 0.08],
        "very_stale_days_threshold": [9999],
        "very_stale_multiplier": [1.00],
        "stale_apply_to_top_profit": [1],
        "low_recent_window": [14],
        "low_recent_threshold": [0, 1],
        "low_recent_multiplier": [0.85, 0.90],
        "low_recent_apply_to_top_profit": [1],
    }

keys = list(PARAM_GRID_RAW.keys())
param_configs = [dict(zip(keys, values)) for values in product(*[PARAM_GRID_RAW[k] for k in keys])]

print("FAST_SEARCH:", FAST_SEARCH)
print("Number of configs:", len(param_configs))
print("First config:")
print(param_configs[0])

## 10. Chạy grid search

Mỗi cấu hình được chấm bằng WRMSSE trên các validation folds. Cấu hình có mean WRMSSE thấp nhất được chọn.

In [ ]:
results = []

for idx, config in enumerate(param_configs, start=1):
    fold_scores = []

    for origin_idx in valid_origins:
        y_true = valid_cache[origin_idx]["y_true"]
        denom = valid_cache[origin_idx]["denom"]
        weights = valid_cache[origin_idx]["weights"]
        y_pred = forecast_matrix_for_origin(config, origin_idx, VALID_HORIZON)
        score = wrmsse_score_cached(y_true, y_pred, denom, weights)
        fold_scores.append(score)

    mean_score = float(np.mean(fold_scores))
    std_score = float(np.std(fold_scores))

    row = config.copy()
    row["mean_wrmsse"] = mean_score
    row["std_wrmsse"] = std_score
    for i, s in enumerate(fold_scores, start=1):
        row[f"fold_{i}_wrmsse"] = float(s)
    results.append(row)

    if idx % 25 == 0 or idx == 1 or idx == len(param_configs):
        print(f"[{idx}/{len(param_configs)}] best so far:", min(r["mean_wrmsse"] for r in results))

results_df = pd.DataFrame(results).sort_values("mean_wrmsse", ascending=True).reset_index(drop=True)
best_config = results_df.iloc[0][keys].to_dict()
best_score = float(results_df.iloc[0]["mean_wrmsse"])

print("Best mean WRMSSE:", best_score)
print("Best config:")
print(json.dumps(best_config, indent=2, ensure_ascii=False))

display(results_df.head(20))
results_df.to_csv(VALID_LOG_PATH, index=False)
print("Saved validation log:", VALID_LOG_PATH)

## 11. Chọn cấu hình tốt nhất

Notebook dùng trực tiếp cấu hình tốt nhất từ grid search để tạo forecast cuối cùng.

In [ ]:
def normalize_config(config: dict) -> dict:
    """Bổ sung key mặc định để forecast function luôn nhận đủ tham số."""
    out = dict(config)
    defaults = {
        "top_weekday_weeks": 8,
        "sparse_cap_multiplier": 1.0,
        "active_cap_window": 56,
        "active_cap_multiplier": 999.0,
        "weekend_multiplier": 1.0,
        "sunday_multiplier": 1.0,
        "month_start_end_multiplier": 1.0,
        "high_season_multiplier": 1.0,
        "eval_window_multiplier": 1.0,
        "stale_days_threshold": 9999,
        "stale_multiplier": 1.0,
        "very_stale_days_threshold": 9999,
        "very_stale_multiplier": 1.0,
        "stale_apply_to_top_profit": 1,
        "low_recent_window": 14,
        "low_recent_threshold": -1,
        "low_recent_multiplier": 1.0,
        "low_recent_apply_to_top_profit": 1,
    }
    for k, v in defaults.items():
        out.setdefault(k, v)
    return out

selected_strategy = "grid_best"
selected_config = normalize_config(best_config)
selected_score = best_score

summary_df = pd.DataFrame([
    {
        "selected_strategy": selected_strategy,
        "mean_wrmsse": selected_score,
        **selected_config,
    }
])

display(summary_df)

print("Selected strategy:", selected_strategy)
print("Selected score:", selected_score)
print("Selected config:")
print(json.dumps(selected_config, indent=2, ensure_ascii=False))


## 12. Forecast 56 ngày tương lai

Dùng cấu hình tốt nhất để dự báo 56 ngày sau ngày cuối cùng trong train.

In [ ]:
final_origin_idx = train_end_idx
future_dates = pd.date_range(all_dates[final_origin_idx] + pd.Timedelta(days=1), periods=FORECAST_HORIZON, freq="D")

final_pred = forecast_matrix_for_origin(selected_config, final_origin_idx, FORECAST_HORIZON)
final_pred = np.clip(final_pred, 0, None).astype(np.float32)

print("Final prediction shape:", final_pred.shape)
print("Forecast period:", future_dates[0].date(), "→", future_dates[-1].date())
print("Selected strategy:", selected_strategy)
print(pd.Series(final_pred.reshape(-1)).describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))

## 13. Tạo submission

Chuyển forecast 56 ngày thành format Kaggle: 28 ngày đầu cho `_validation`, 28 ngày sau cho `_evaluation`.

In [ ]:
forecast_cols = [f"F{i}" for i in range(1, PUBLIC_HORIZON + 1)]
final_pred = np.asarray(final_pred, dtype=np.float32)

assert final_pred.shape[0] == len(all_skus), (
    f"Số SKU trong final_pred ({final_pred.shape[0]}) khác all_skus ({len(all_skus)})"
)
assert final_pred.shape[1] == FORECAST_HORIZON, (
    f"Số horizon trong final_pred ({final_pred.shape[1]}) khác FORECAST_HORIZON ({FORECAST_HORIZON})"
)

pred_validation = pd.DataFrame(
    final_pred[:, :PUBLIC_HORIZON],
    index=pd.Index(all_skus, name="ItemCode"),
    columns=forecast_cols,
)

pred_evaluation = pd.DataFrame(
    final_pred[:, PUBLIC_HORIZON:FORECAST_HORIZON],
    index=pd.Index(all_skus, name="ItemCode"),
    columns=forecast_cols,
)

submission = sample_sub[["id"]].copy()
submission["ItemCode"] = (
    submission["id"]
    .str.replace("_validation", "", regex=False)
    .str.replace("_evaluation", "", regex=False)
)

is_validation = submission["id"].str.endswith("_validation")
is_evaluation = submission["id"].str.endswith("_evaluation")

invalid_ids = submission.loc[~(is_validation | is_evaluation), "id"].tolist()
if len(invalid_ids) > 0:
    raise ValueError(f"Có id không hợp lệ: {invalid_ids[:10]}")

missing_skus = sorted(set(submission["ItemCode"]) - set(all_skus))
if len(missing_skus) > 0:
    print(f"Cảnh báo: Có {len(missing_skus)} SKU thiếu trong prediction. Các SKU này sẽ được fill 0.")
    print(missing_skus[:10])

submission.loc[is_validation, forecast_cols] = (
    pred_validation
    .reindex(submission.loc[is_validation, "ItemCode"])
    .fillna(0)
    .to_numpy()
)

submission.loc[is_evaluation, forecast_cols] = (
    pred_evaluation
    .reindex(submission.loc[is_evaluation, "ItemCode"])
    .fillna(0)
    .to_numpy()
)

submission = submission.drop(columns=["ItemCode"])
submission[forecast_cols] = submission[forecast_cols].astype("float32").clip(lower=0).fillna(0)

assert submission.shape == sample_sub.shape, f"Shape sai: submission {submission.shape}, sample {sample_sub.shape}"
assert submission["id"].tolist() == sample_sub["id"].tolist(), "Thứ tự id không khớp sample_submission"
assert submission["id"].duplicated().sum() == 0, "Có duplicate id"
assert submission[forecast_cols].isna().sum().sum() == 0, "Có NaN trong forecast"
assert (submission[forecast_cols].values < 0).sum() == 0, "Có forecast âm"

submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved submission:", SUBMISSION_PATH)
print("Shape:", submission.shape)
print("Validation-window forecast sum:", float(submission.loc[submission["id"].str.endswith("_validation"), forecast_cols].values.sum()))
print("Evaluation-window forecast sum:", float(submission.loc[submission["id"].str.endswith("_evaluation"), forecast_cols].values.sum()))
display(submission.head())

## 14. Lưu cấu hình tốt nhất

Lưu cấu hình được chọn và thông tin validation để tái lập kết quả.

In [ ]:
best_payload = {
    "selected_strategy": selected_strategy,
    "selected_config": selected_config,
    "selected_mean_wrmsse": selected_score,
    "grid_best_config": normalize_config(best_config),
    "grid_best_mean_wrmsse": best_score,
    "fast_search": FAST_SEARCH,
    "valid_fold_offsets": VALID_FOLD_OFFSETS,
    "valid_horizon": VALID_HORIZON,
    "seed": SEED,
}

with open(BEST_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, ensure_ascii=False, indent=2)

print("Saved best config:", BEST_CONFIG_PATH)
print(json.dumps(best_payload, indent=2, ensure_ascii=False))


## 15. File output

File nộp Kaggle nằm tại `outputs/submission.csv`. Các file log dùng để kiểm tra lại quá trình validation.